In [9]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [10]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-11-12 14:16:58.965020-05:00


#### 4.1) `assemble cluster features` (+ robust scaling)

In [11]:
# This only prepares the matrix; we will cluster the top anomalies after 4.2 scoring.
from sklearn.preprocessing import RobustScaler

q = """
SELECT fp.provider,
       fp.avg_reimb_per_claim, fp.claims_30d, fp.unique_benes_30d, fp.px_distinct_30d,
       fp.ip_avg_z_los_365, fp.ip_avg_z_reimb_365, fp.ip_max_z_los_365, fp.ip_max_z_reimb_365,
       fp.op_avg_z_reimb_365, fp.op_max_z_reimb_365,
       cm.ip_share, cm.op_share
FROM mart.features_provider fp
LEFT JOIN mart.provider_claim_mix cm USING (provider)
"""
provX = pd.read_sql(q, con=engine).fillna(0)

id_col = "provider"
feat_cols = [c for c in provX.columns if c != id_col]

# median/IQR scaling
Xs = RobustScaler().fit_transform(provX[feat_cols])

cluster_input = pd.DataFrame(Xs, columns=feat_cols)
cluster_input[id_col] = provX[id_col]

# Persist for 4.4 (HDBSCAN after we have anomaly top-X from 4.2)
cluster_input.to_sql("cluster_input_provider", con=engine, schema="mart",
                     if_exists="replace", index=False)

410

In [12]:
# IP features (include LOS + IP peer z's)
q_ip = """
SELECT fc.claimid, fc.provider, fc.claim_type,
       fc.reimb_amt, fc.deductible_paid, fc.dx_count, fc.px_count, fc.los_days,
       z.z_ip_reimb, z.z_ip_los
FROM mart.features_claim fc
LEFT JOIN mart.rule_claim_z z USING (claimid)
WHERE fc.claim_type = 'IP';
"""
ip = pd.read_sql(q_ip, con=engine)

# OP features (no LOS; use OP peer z)
q_op = """
SELECT fc.claimid, fc.provider, fc.claim_type,
       fc.reimb_amt, fc.deductible_paid, fc.dx_count, fc.px_count,
       z.z_op_reimb
FROM mart.features_claim fc
LEFT JOIN mart.rule_claim_z z USING (claimid)
WHERE fc.claim_type = 'OP';
"""
op = pd.read_sql(q_op, con=engine)


4.2) Train unsupervised models: `IsolationForest`, `LocalOutlierFactor`, `One-Class SVM` on provider-level features.


A) `Load claim features` (IP & OP)


B) Train + score `(IF, LOF, OCSVM)` per claim type


`IF: tree-based`; we'll use `score_samples` (flip sign so higher = more anomalous). `contamination` tunes the default cutoff later.

`LOF: for training-set` scoring use default `novelty=False` and read `negative_outlier_factor_` (more negative = more outlier). For future/test scoring use a separate model with `novelty=True`.

`OCSVM: scale first`; use -decision_function so higher = more anomalous. 

`Scaling: use RobustScaler` (median/IQR) for LOF/OCSVM; OK to keep IF unscaled.


In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

def fit_score_models(df, feat_cols, contamination=0.01, nu=0.01):
    out_frames = []
    
    # ---------- IsolationForest ----------
    X = df[feat_cols].copy()
    X_if = X.fillna(X.median(numeric_only=True))
    iforest = IsolationForest(
        n_estimators=400, contamination=contamination,
        random_state=42, n_jobs=-1
    ).fit(X_if)
    
    # higher = more anomalous (flip sign)
    score_if = -iforest.score_samples(X_if)
    out_frames.append(pd.DataFrame(
        {"claimid": df["claimid"], "model": "IF", "anomaly_score": score_if}))

    # ---------- LOF (train-time scoring) ----------
    # robust to outliers
    X_lof = RobustScaler().fit_transform(X.fillna(X.median(numeric_only=True)))
    # novelty=False (default)
    lof = LocalOutlierFactor(n_neighbors=35, contamination=contamination)
    lof.fit_predict(X_lof)  # computes negative_outlier_factor_
    score_lof = -lof.negative_outlier_factor_  # higher = more anomalous
    out_frames.append(pd.DataFrame(
        {"claimid": df["claimid"], "model": "LOF", "anomaly_score": score_lof}))

    # (Optional) keep a novelty=True LOF for future/test scoring
    lof_novel = LocalOutlierFactor(n_neighbors=35, novelty=True).fit(X_lof)

    # ---------- One-Class SVM ----------
    X_svm = RobustScaler().fit_transform(X.fillna(X.median(numeric_only=True)))
    # nu ~ expected outlier fraction
    ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    ocsvm.fit(X_svm)
    # higher = more anomalous
    score_svm = -ocsvm.decision_function(X_svm)
    out_frames.append(pd.DataFrame(
        {"claimid": df["claimid"], "model": "OCSVM", "anomaly_score": score_svm}))

    scores = pd.concat(out_frames, ignore_index=True)
    return scores, {"iforest": iforest, "lof_novel": lof_novel, "ocsvm": ocsvm}

# Feature sets
ip_feats = ["reimb_amt", "deductible_paid", "dx_count",
            "px_count", "los_days", "z_ip_reimb", "z_ip_los"]
op_feats = ["reimb_amt", "deductible_paid",
            "dx_count", "px_count", "z_op_reimb"]

ip_scores, ip_models = fit_score_models(ip, ip_feats, contamination=0.01, nu=0.01)
op_scores, op_models = fit_score_models(op, op_feats, contamination=0.01, nu=0.01)

scores = (pd.concat([ip_scores.assign(claim_type="IP"),
                     op_scores.assign(claim_type="OP")], ignore_index=True))


c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\neighbors\_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


C) `Persist scores` + handy ranks


In [ ]:
# Rank within claim_type + model (percentile, higher = more anomalous)
scores["rank_pct"] = scores.groupby(["claim_type", "model"])["anomaly_score"] \
                           .rank(pct=True, ascending=True)  # since higher score = more anomalous

# Write long-form table to DB
scores.to_sql("anomaly_claim_scores", con=engine, schema="mart",
              if_exists="replace", index=False)


NameError: name 'scores' is not defined

#### 4.3) Score all providers/claims with anomaly scores.


A) `Rank & consensus`


In [ ]:
# columns expected: claimid, model ∈ {IF, LOF, OCSVM}, claim_type ∈ {IP, OP}, anomaly_score
scores = pd.read_sql("SELECT * FROM mart.anomaly_claim_scores", con=engine)

# 1) Percentile rank per claim_type & model, DESC so top anomalies get rank_pct≈1.0
scores["rank_pct"] = scores.groupby(["claim_type","model"])["anomaly_score"] \
                           .rank(pct=True, ascending=False)

# 2) Pivot to compute a simple consensus (mean of percentiles across models)
wide = scores.pivot_table(index=["claimid","claim_type"], columns="model",
                          values="rank_pct", aggfunc="first")
# If a detector is missing for a claim, treat as neutral 0.5
wide = wide.reindex(columns=["IF","LOF","OCSVM"]).fillna(0.5)
wide["consensus_rank"] = wide[["IF","LOF","OCSVM"]].mean(axis=1)

# 3) Back to long form with consensus added
ranked = scores.merge(
    wide["consensus_rank"].reset_index(),
    on=["claimid","claim_type"], how="left"
)

# 4) Persist ranked scores
ranked.to_sql("anomaly_claim_scores_ranked", con=engine, schema="mart",
              if_exists="replace", index=False)


InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table anomaly_claim_scores_ranked because other objects depend on it
DETAIL:  materialized view anomaly_claim_top_pctl depends on table anomaly_claim_scores_ranked
materialized view anomaly_claim_topk depends on table anomaly_claim_scores_ranked
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE mart.anomaly_claim_scores_ranked]
(Background on this error at: https://sqlalche.me/e/20/2j85)

#### 4.4) Clustering top anomalies with HDBSCAN


A) `Pull top anomalies` & build the `feature matrix`


In [ ]:
from sklearn.preprocessing import RobustScaler  # robust to outliers

# 1) Load top anomalies
top = pd.read_sql(
    "SELECT claimid, claim_type FROM mart.anomaly_claim_top_pctl", con=engine)
# if using topk: SELECT claimid, claim_type FROM mart.anomaly_claim_topk

# 2) Join back to features
q_feats = """
SELECT claimid, claim_type,
       reimb_amt, deductible_paid, dx_count, px_count,
       los_days, z_ip_reimb, z_ip_los, z_op_reimb
FROM mart.features_claim
"""
feats = pd.read_sql(q_feats, con=engine)

X = top.merge(feats, on=["claimid", "claim_type"], how="left")

# 3) Build per-cohort matrices + robust scale
def prep_matrix(df, cohort):
    dfc = df[df["claim_type"].eq(cohort)].copy()
    if cohort == "IP":
        cols = ["reimb_amt", "deductible_paid", "dx_count",
                "px_count", "los_days", "z_ip_reimb", "z_ip_los"]
    else:
        cols = ["reimb_amt", "deductible_paid",
                "dx_count", "px_count", "z_op_reimb"]
    M = dfc[cols].fillna(dfc[cols].median(numeric_only=True))
    
    # median/IQR scaling helps with heavy tails
    Ms = RobustScaler().fit_transform(M)
    dfc = dfc[["claimid", "claim_type"]].copy()
    return dfc, Ms, cols


ip_ids, X_ip, ip_cols = prep_matrix(X, "IP")
op_ids, X_op, op_cols = prep_matrix(X, "OP")


NameError: name 'pd' is not defined

B) Run `HDBSCAN` on each cohort's `top anomalies`


In [ ]:
import hdbscan

def run_hdbscan(X, min_cluster_size=25, min_samples=None):
    # 'eom' selects the most persistent clusters from the condensed tree
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                                min_samples=min_samples,
                                cluster_selection_method="eom",
                                prediction_data=True)  # enables probabilities for later use
    labels = clusterer.fit_predict(X)
    probs = clusterer.probabilities_              # soft membership 0..1
    outlier_scores = clusterer.outlier_scores_     # higher = more outlier-like
    return clusterer, labels, probs, outlier_scores


ip_model, ip_lab, ip_prob, ip_out = run_hdbscan(X_ip, min_cluster_size=25)
op_model, op_lab, op_prob, op_out = run_hdbscan(X_op, min_cluster_size=25)

ip_clusters = ip_ids.assign(
    cluster=ip_lab, membership_prob=ip_prob, outlier_score=ip_out)
op_clusters = op_ids.assign(
    cluster=op_lab, membership_prob=op_prob, outlier_score=op_out)

clusters = pd.concat([ip_clusters, op_clusters], ignore_index=True)
clusters.to_sql("anomaly_claim_clusters", con=engine, schema="mart",
                if_exists="replace", index=False)


c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\hdbscan\hdbscan_.py:1489: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = 

344

C) `Quick cluster` profiles


In [ ]:
# Join cluster labels back to features for profiling
prof = clusters.merge(feats, on=["claimid", "claim_type"], how="left")


def profile(df, cohort, cols):
    sub = df[(df["claim_type"] == cohort) & (df["cluster"] >= 0)]
    g = sub.groupby("cluster")
    size = g.size().rename("n")
    med = g[cols].median().add_prefix("med_")
    return size.to_frame().join(med).reset_index()


ip_profile = profile(prof, "IP", ip_cols)
op_profile = profile(prof, "OP", op_cols)

# Save
ip_profile.to_sql("anomaly_claim_cluster_profiles_ip", con=engine, schema="mart",
                  if_exists="replace", index=False)
op_profile.to_sql("anomaly_claim_cluster_profiles_op", con=engine, schema="mart",
                  if_exists="replace", index=False)


50

D) `Visualize high-quality` clusters


In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np


def umap_plot(X, labels, title):
    emb = umap.UMAP(n_neighbors=15, min_dist=0.05,
               random_state=42).fit_transform(X)
    plt.figure(figsize=(7, 6))
    # plot noise in light gray
    noise = labels < 0
    plt.scatter(emb[noise, 0], emb[noise, 1], s=6, alpha=0.3, label="noise")
    # plot clusters
    for c in np.unique(labels[~noise]):
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1], s=10, label=f"cluster {c}")
    plt.title(title)
    plt.legend(markerscale=2)
    plt.show()


umap_plot(X_ip, ip_lab, "HDBSCAN clusters (IP top anomalies)")
umap_plot(X_op, op_lab, "HDBSCAN clusters (OP top anomalies)")


NameError: name 'X_ip' is not defined

They show (a) how many distinct groups HDBSCAN found, (b) how separated they are in the feature space, and (c) how much noise (unclustered one-offs) you have. That’s all good and expected. UMAP is for visualization, not the clustering itself

IP (6 clusters + some noise): inpatient anomalies coalesce into a few pattern families—good sign that IP anomalies are more homogeneous.

OP (~50 small clusters + noise): outpatient anomalies are much more diverse (common in OP). Many small, stable groups are normal for HDBSCAN when the data mix is rich. You can merge by raising min_cluster_size/using eom selection or by sub-cohorting (e.g., by APC family) before clustering.